In [8]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [9]:
def find_repo_root(start=None):
    """Walk up from current working directory until the baseline dataset is found."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "Dataset" / "students_mental_health_survey_with_burnout_final.csv").exists():
            return cand
    raise FileNotFoundError("Could not locate repository root containing Dataset/students_mental_health_survey_with_burnout_final.csv")

ROOT_PATH = find_repo_root()
ROOT = str(ROOT_PATH)
DATA_PATH = str(ROOT_PATH / "Dataset" / "students_mental_health_survey_with_burnout_final.csv")

df = pd.read_csv(DATA_PATH)
print('Dataset: Dataset/students_mental_health_survey_with_burnout_final.csv')
print('Shape:', df.shape)

Dataset: Dataset/students_mental_health_survey_with_burnout_final.csv
Shape: (10000, 29)


In [10]:
df = pd.read_csv(find_repo_root() / "Dataset" / "students_mental_health_survey_with_burnout_final.csv")
df.head()

,Age,Course,Gender,CGPA,Stress_Level,Depression_Score,Anxiety_Score,Sleep_Quality,Physical_Activity,Diet_Quality,...,Residence_Type,burnout_composite_score,burnout,burnout_raw_score,method1_tertiles,method2_wider,method3_very_wide,method4_manual,method5_manual2,method6_kmeans
0,29,Medical,Female,3.69,5,0,3,Good,Low,Average,...,On-Campus,0.199412,0,2.666667,1,1,1,1,1,1
1,24,Business,Female,3.75,1,3,0,Good,Low,Average,...,On-Campus,-0.610848,0,1.333333,0,0,1,0,0,0
2,25,Engineering,Male,3.15,3,2,4,Good,Moderate,Average,...,Off-Campus,0.408598,0,3.000000,2,2,2,1,1,2
3,18,Law,Male,3.97,1,1,5,Poor,Moderate,Average,...,With Family,0.001607,0,2.333333,1,1,1,1,1,1
4,25,Law,Female,3.13,4,3,5,Average,Moderate,Average,...,On-Campus,1.021761,1,4.000000,2,2,2,2,2,2


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [12]:
df2 = pd.read_csv(DATA_PATH)

In [13]:
df2

,Age,Course,Gender,CGPA,Stress_Level,Depression_Score,Anxiety_Score,Sleep_Quality,Physical_Activity,Diet_Quality,...,Residence_Type,burnout_composite_score,burnout,burnout_raw_score,method1_tertiles,method2_wider,method3_very_wide,method4_manual,method5_manual2,method6_kmeans
0,29,Medical,Female,3.69,5,0,3,Good,Low,Average,...,On-Campus,0.199412,0,2.666667,1,1,1,1,1,1
1,24,Business,Female,3.75,1,3,0,Good,Low,Average,...,On-Campus,-0.610848,0,1.333333,0,0,1,0,0,0
2,25,Engineering,Male,3.15,3,2,4,Good,Moderate,Average,...,Off-Campus,0.408598,0,3.000000,2,2,2,1,1,2
3,18,Law,Male,3.97,1,1,5,Poor,Moderate,Average,...,With Family,0.001607,0,2.333333,1,1,1,1,1,1
4,25,Law,Female,3.13,4,3,5,Average,Moderate,Average,...,On-Campus,1.021761,1,4.000000,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,20,Business,Male,3.51,0,3,2,Average,Low,Average,...,With Family,-0.404676,0,1.666667,0,1,1,0,1,0
9996,23,Medical,Male,3.40,4,5,3,Good,Moderate,Good,...,Off-Campus,1.023250,1,4.000000,2,2,2,2,2,2
9997,23,Computer Science,Male,3.91,1,4,4,Poor,High,Average,...,Off-Campus,0.413135,1,3.000000,2,2,2,1,1,2
9998,22,Others,Female,3.66,5,0,2,Average,Moderate,Poor,...,With Family,-0.005235,0,2.333333,1,1,1,1,1,1


In [14]:
# Define bins for 0–5 scale
bins = [0, 1.25, 2.5, 3.75, 5]
labels = ["0", "1", "2", "3"]

# Create category column
df2["burnout_category"] = pd.cut(
    df2["burnout_raw_score"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [15]:
df2
df.columns

Index(['Age', 'Course', 'Gender', 'CGPA', 'Stress_Level', 'Depression_Score',
       'Anxiety_Score', 'Sleep_Quality', 'Physical_Activity', 'Diet_Quality',
       'Social_Support', 'Relationship_Status', 'Substance_Use',
       'Counseling_Service_Use', 'Family_History', 'Chronic_Illness',
       'Financial_Stress', 'Extracurricular_Involvement',
       'Semester_Credit_Load', 'Residence_Type', 'burnout_composite_score',
       'burnout', 'burnout_raw_score', 'method1_tertiles', 'method2_wider',
       'method3_very_wide', 'method4_manual', 'method5_manual2',
       'method6_kmeans'],
      dtype='str')

## First attempt at random forest below

In [16]:
import numpy as np

## Attempt with SMOTE below

## Trying to find best hyperparameters below

## With SMOTE and best hyperparameters below

## Used same hyperparameters as above but removed SMOTE below

## Only 3 classes start

## Adding SMOTE

In [17]:
from imblearn.over_sampling import SMOTE

df5 = pd.read_csv(DATA_PATH)

# =========================
# 2. CREATE TARGET (3 EQUAL-WIDTH BINS)
# =========================
labels = ["low burnout", "mid burnout", "high burnout"]

df5["burnout_category"] = pd.cut(
    df5["burnout_raw_score"],
    bins=3,
    labels=labels,
    include_lowest=True
)

# =========================
# 3. HOT-DECK IMPUTATION
# =========================
def stochastic_hotdeck(series, seed=42):
    np.random.seed(seed)

    observed = series.dropna().values
    if len(observed) == 0:
        return series

    missing = series.isna()

    series.loc[missing] = np.random.choice(
        observed,
        size=missing.sum(),
        replace=True
    )

    return series


for col in df5.columns:
    if col != "burnout_category":
        df5[col] = stochastic_hotdeck(df5[col])


# =========================
# 4. FEATURES / TARGET
# =========================
y = df5["burnout_category"]

X = df5.drop(columns=[
    "burnout",
    "burnout_raw_score",
    "burnout_composite_score",
    "burnout_category",
    "Depression_Score",
    "Stress_Level",
    "Anxiety_Score", 'method1_tertiles', 'method2_wider',
       'method3_very_wide', 'method4_manual', 'method5_manual2',
       'method6_kmeans'
])


# =========================
# 5. ENCODING
# =========================
label_encoders = {}

for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

y = LabelEncoder().fit_transform(y)


# =========================
# 6. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# 7. APPLY SMOTE (ONLY TRAINING DATA)
# =========================
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)


# =========================
# 8. RANDOM FOREST
# =========================
model = RandomForestClassifier(
    n_estimators=300,
    max_features="log2",
    min_samples_split=5,
    min_samples_leaf=1,
    max_depth=None,
    random_state=42
)

model.fit(X_train_sm, y_train_sm)


# =========================
# 9. PREDICT
# =========================
y_pred = model.predict(X_test)


# =========================
# 10. EVALUATION
# =========================
print(classification_report(y_test, y_pred))

C:\Users\takbh\AppData\Local\Temp\ipykernel_2464\3191508084.py:66: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object"]).columns:


              precision    recall  f1-score   support

           0       0.14      0.12      0.13       216
           1       0.30      0.21      0.25       606
           2       0.60      0.70      0.65      1178

    accuracy                           0.49      2000
   macro avg       0.35      0.35      0.34      2000
weighted avg       0.46      0.49      0.47      2000



## Testing different hyperparameters

## No Smote with hyperparameter search

In [18]:
import joblib
from sklearn.metrics import confusion_matrix

# Define output directory
output_dir = Path(ROOT) / "ml_randomforest" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save the best model
#model_path = output_dir / "best_random_forest_model.joblib"
#joblib.dump(model, model_path)
#print(f"Model saved to: {model_path}")

# Save predictions
predictions_df = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
})
predictions_path = output_dir / "predictions.csv"
predictions_df.to_csv(predictions_path, index=False)
print(f"Predictions saved to: {predictions_path}")

# Save classification report
report_path = output_dir / "classification_report.txt"
with open(report_path, "w") as f:
    f.write(classification_report(y_test, y_pred))
print(f"Classification report saved to: {report_path}")

# Save confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_path = output_dir / "confusion_matrix.csv"
cm_df.to_csv(cm_path)
print(f"Confusion matrix saved to: {cm_path}")

# Save best parameters
best_params = {
    "n_estimators": 300,
    "max_features": "log2",
    "min_samples_split": 5,
    "min_samples_leaf": 1,
    "max_depth": None,
    "random_state": 42
}
params_path = output_dir / "best_parameters.txt"
with open(params_path, "w") as f:
    f.write(str(best_params))
print(f"Best parameters saved to: {params_path}")

print("All outputs saved successfully!")

Predictions saved to: C:\Users\takbh\auracheck\ml_randomforest\outputs\predictions.csv
Classification report saved to: C:\Users\takbh\auracheck\ml_randomforest\outputs\classification_report.txt
Confusion matrix saved to: C:\Users\takbh\auracheck\ml_randomforest\outputs\confusion_matrix.csv
Best parameters saved to: C:\Users\takbh\auracheck\ml_randomforest\outputs\best_parameters.txt
All outputs saved successfully!


In [19]:
import json
from sklearn.metrics import accuracy_score

# Reuse the same output directory used above
output_dir = Path(ROOT) / "ml_randomforest" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

summary_payload = {
    "best_parameter": best_params,
    "output": {
        "sample_predictions": predictions_df.head(20).to_dict(orient="records"),
        "total_predictions": int(len(predictions_df)),
        "accuracy": float(accuracy_score(y_test, y_pred))
    },
    "evaluation_matrix": classification_report(y_test, y_pred, output_dict=True),
    "confusion_matrix": {
        "labels": labels,
        "matrix": cm.tolist()
    }
}

json_path = output_dir / "random_forest_outputs_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print(f"JSON summary saved to: {json_path}")

JSON summary saved to: C:\Users\takbh\auracheck\ml_randomforest\outputs\random_forest_outputs_summary.json
